In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/fingerprint_mlp_fold0.pth
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/test_predicted_fingerprints.npy


In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 41.3 MB/s  0:00:00 eta 0:00:01


# 1. Importaciones, Configuración de Entorno y Rutas

In [3]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from tqdm import tqdm

# --- CONFIGURACIÓN DE RUTAS ---
DATA_DIR = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
TRAIN_FOLDS_PATH = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet'
TRAIN_PATH = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet'

# Dataset donde guardaste los pesos del modelo del Notebook 2
MODEL_DIR = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints'
MODEL_PATH = os.path.join(MODEL_DIR, 'fingerprint_mlp_fold0.pth')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_FOLD = 0  # Usaremos el Fold 0 como conjunto de validación

# 2. Definición de Arquitectura y Dataset de Validación

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(0.2), # Dropout moderado para evitar sobreajuste rápido
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        # Conexión residual: la entrada se suma a la salida del bloque
        return self.relu(x + self.block(x))

class TopKEncoderMLP(nn.Module):
    def __init__(self, top_k=64, input_dim=2, hidden_dim=512, output_dim=1024):
        super().__init__()
        self.top_k = top_k
        
        # Proyección inicial por cada pico (BatchNorm sobre hidden_dim, no sobre top_k)
        self.pico_mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim), 
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Capas globales para predecir el fingerprint de 1024 bits
        self.global_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, output_dim) # Devuelve logits
        )

    def forward(self, x):
            # x tiene forma: (Batch_Size, Top_K, 2) [m/z, intensity]
            batch_size = x.size(0)
            
            # Aplanar para aplicar la capa lineal a todos los picos a la vez: (Batch_Size * Top_K, input_dim)
            x_flat = x.view(-1, x.size(-1))
            out_picos = self.pico_mlp(x_flat)
            
            # Reestructurar de vuelta a (Batch_Size, Top_K, hidden_dim)
            out_picos = out_picos.view(batch_size, self.top_k, -1)
            
            # Agregación global (Max Pooling a lo largo de los picos para capturar los fragmentos clave)
            global_repr, _ = torch.max(out_picos, dim=1)
            
            # Predicción final de los 1024 bits
            logits = self.global_mlp(global_repr)
            return logits

# Dataset basado en Top-K picos que calcula el fingerprint al vuelo de forma ligera por índice
class TopKTestDataset(Dataset):
    def __init__(self, df, top_k=64):
        self.df = df.reset_index(drop=True)
        self.top_k = top_k

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Procesar picos espectrales Top-K de test
        mz = np.array(row['ms2_mzs'], dtype=np.float32)
        intensity = np.array(row['ms2_normalized_intensities'], dtype=np.float32)
        
        if len(mz) > 0:
            sorted_idx = np.argsort(intensity)[::-1]
            mz = mz[sorted_idx][:self.top_k]
            intensity = intensity[sorted_idx][:self.top_k]
            
            max_int = intensity.max()
            if max_int > 0:
                intensity = intensity / max_int
        
        current_len = len(mz)
        if current_len < self.top_k:
            mz = np.pad(mz, (0, self.top_k - current_len), 'constant', constant_values=0)
            intensity = np.pad(intensity, (0, self.top_k - current_len), 'constant', constant_values=0)
            
        spectrum_features = np.stack([mz, intensity], axis=-1)
        
        # Retorna únicamente x (el tensor de picos), evitando buscar 'normalized_smiles'
        return torch.tensor(spectrum_features, dtype=torch.float32)

# 3. Cargar Folds y Separar Validación

In [8]:
print("Cargando dataset original de espectros y folds asignados...")
full_train_df = pd.read_parquet(TRAIN_PATH)
folds_df = pd.read_parquet(TRAIN_FOLDS_PATH)

if 'fold' in full_train_df.columns:
    full_train_df = full_train_df.drop(columns=['fold'])

# Identificar columnas clave de forma robusta
col_smiles = 'normalized_smiles' if 'normalized_smiles' in folds_df.columns else 'smiles'
id_col = 'inchikey14' if 'inchikey14' in folds_df.columns else 'inchikey'

# Hacer el merge para que el dataframe resultante tenga tanto los espectros como el fold asignado
train_folds_df = full_train_df.merge(
    folds_df[[col_smiles, id_col, 'fold']].drop_duplicates(),
    on=[col_smiles, id_col],
    how='inner'
)

# Separar en Train y Validación local (Fold 0)
val_df = train_folds_df[train_folds_df['fold'] == EVAL_FOLD].reset_index(drop=True)
train_df = train_folds_df[train_folds_df['fold'] != EVAL_FOLD].reset_index(drop=True)

print(f"Espectros para Entrenamiento: {len(train_df):,}")
print(f"Espectros de validación en Fold {EVAL_FOLD}: {len(val_df):,}")

# Identificar moléculas únicas en validación y sus metadatos
unique_val_mols = val_df[id_col].unique()
val_ground_truth = val_df.groupby(id_col)[col_smiles].first().to_dict()
val_precursor_mzs = val_df.groupby(id_col)['precursor_mz'].first().to_dict()

print(f"Moléculas únicas en validación: {len(unique_val_mols):,}")

# Liberar memoria de los dataframes gigantes combinados
del train_folds_df, folds_df
import gc; gc.collect()

Cargando dataset original de espectros y folds asignados...
Espectros para Entrenamiento: 2,031,686
Espectros de validación en Fold 0: 507,922
Moléculas únicas en validación: 55,162


0

# 4. Inferencia sobre el Fold de Validación con el Modelo Entrenado

In [6]:
print("Cargando modelo neuronal entrenado en Fase 2...")
val_dataset = TopKTestDataset(val_df)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)

# Dimensión de entrada: num_bins (1000.0 / 0.5 = 2000) + 1 (precursor_mz)
INPUT_DIM = int(1000.0 / 0.5) + 1
OUTPUT_DIM = 1024  # Tamaño del Morgan Fingerprint

model = TopKEncoderMLP(top_k=64, input_dim=2, hidden_dim=512, output_dim=1024).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

pred_fps_list = []
with torch.no_grad():
    for x_batch in tqdm(val_loader, desc="Inferencia de Fingerprints"):
        # ¡Mover el batch al dispositivo (GPU o CPU) configurado en DEVICE!
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        
        with torch.amp.autocast('cuda' if 'cuda' in str(DEVICE) else 'cpu'):
            logits = model(x_batch)
            preds = torch.sigmoid(logits)
            
        pred_fps_list.append(preds.detach().cpu().numpy())

val_preds = np.concatenate(pred_fps_list, axis=0)

# Agrupar y promediar fingerprints por molecule_id
val_df['fp_idx'] = np.arange(len(val_df))
mol_to_fp_val = {}

for mol_id, group in val_df.groupby(id_col):
    indices = group['fp_idx'].values
    avg_fp = val_preds[indices].mean(axis=0)
    mol_to_fp_val[mol_id] = avg_fp

del val_preds
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("✓ Predicciones de validación listas.")

Cargando modelo neuronal entrenado en Fase 2...


Inferencia de Fingerprints: 100%|██████████| 1985/1985 [01:04<00:00, 30.91it/s]


✓ Predicciones de validación listas.


# 5: Construcción de la Matriz de Candidatos Completa

In [9]:
print("Construyendo librería de candidatos desde el dataset completo...")
candidate_df = full_train_df.drop_duplicates(subset=['inchikey14']).reset_index(drop=True)

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
num_candidates = len(candidate_df)
candidate_fps_matrix = np.zeros((num_candidates, 1024), dtype=np.uint8)
candidate_mzs = np.zeros(num_candidates, dtype=np.float32)
valid_smiles = []

valid_count = 0
for idx, row in tqdm(candidate_df.iterrows(), total=num_candidates, desc="Indexando Candidatos"):
    smi = str(row[col_smiles]) if pd.notnull(row[col_smiles]) else ""
    if not smi:
        continue
    
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = morgan_gen.GetFingerprint(mol)
        candidate_fps_matrix[valid_count] = np.frombuffer(fp.ToBitString().encode('utf-8'), 'u1') - ord('0')
        candidate_mzs[valid_count] = row.get('precursor_mz', 0.0)
        valid_smiles.append(smi)
        valid_count += 1

candidate_fps_matrix = candidate_fps_matrix[:valid_count]
candidate_mzs = candidate_mzs[:valid_count]
candidate_df = pd.DataFrame({'smiles': valid_smiles, 'precursor_mz': candidate_mzs})

print(f"✓ Total candidatos indexados: {len(candidate_df):,}")

Construyendo librería de candidatos desde el dataset completo...


Indexando Candidatos: 100%|██████████| 275810/275810 [02:01<00:00, 2267.14it/s]


✓ Total candidatos indexados: 275,810


# 6: Cálculo de la Métrica Oficial MRR@25 con Filtrado por Masa

In [10]:
def cosine_similarity(query_fp, candidate_matrix):
    # Similitud coseno vectorizada entre el vector continuo predicho y los candidatos
    dot_product = np.dot(candidate_matrix, query_fp)
    query_norm = np.linalg.norm(query_fp)
    cand_norms = np.linalg.norm(candidate_matrix, axis=1)
    
    denom = cand_norms * query_norm
    scores = np.divide(dot_product, denom, out=np.zeros_like(dot_product, dtype=float), where=denom!=0)
    return np.nan_to_num(scores, nan=0.0)

def compute_mrr_at_25_improved(query_mol_ids, mol_to_fp, mol_to_mz, ground_truth, candidate_fps, candidate_mzs, candidate_df, ppm_tolerance=20.0):
    reciprocal_ranks = []
    top_k = 25
    top1_hits = 0
    top5_hits = 0
    top25_hits = 0

    for mol_id in tqdm(query_mol_ids, desc="Evaluando MRR@25 (Mejorado)"):
        true_smiles = ground_truth[mol_id]
        pred_vector = mol_to_fp[mol_id]  # Vector continuo de probabilidades (0 a 1)
        target_mz = mol_to_mz[mol_id]
        
        # Filtro de masa basado en ppm (Partes por Millón)
        # delta_ppm = |m_cand - target_mz| / target_mz * 1e6
        if target_mz > 0:
            ppm_diff = np.abs(candidate_mzs - target_mz) / target_mz * 1e6
            mz_mask = ppm_diff <= ppm_tolerance
            filtered_indices = np.where(mz_mask)[0]
            
            # Si la ventana en ppm es muy estricta y hay pocos candidatos, ampliar a 50 ppm
            if len(filtered_indices) < top_k:
                mz_mask = ppm_diff <= 50.0
                filtered_indices = np.where(mz_mask)[0]
        else:
            filtered_indices = np.arange(len(candidate_fps))
            
        if len(filtered_indices) == 0:
            reciprocal_ranks.append(0.0)
            continue
            
        sub_matrix = candidate_fps[filtered_indices]
        # Usamos similitud Coseno sobre los vectores
        scores = cosine_similarity(pred_vector, sub_matrix)
        
        # Extraer Top-25
        fetch_k = min(top_k, len(scores))
        top_sub_indices = np.argpartition(scores, -fetch_k)[-fetch_k:]
        top_sub_indices = top_sub_indices[np.argsort(scores[top_sub_indices])[::-1]]
        
        retrieved_smiles = candidate_df.iloc[filtered_indices[top_sub_indices]]['smiles'].tolist()
        
        # Verificar si el SMILES verdadero está en las predicciones
        if true_smiles in retrieved_smiles:
            rank = retrieved_smiles.index(true_smiles) + 1  # Rango 1-indexed
            reciprocal_ranks.append(1.0 / rank)
            if rank == 1:
                top1_hits += 1
            if rank <= 5:
                top5_hits += 1
            if rank <= 25:
                top25_hits += 1
        else:
            reciprocal_ranks.append(0.0)
            
    mrr_score = np.mean(reciprocal_ranks)
    total = len(query_mol_ids)
    
    print("\n" + "="*50)
    print("  RESULTADOS DE VALIDACIÓN LOCAL (MEJORADO - COCOS/PPM)  ")
    print("="*50)
    print(f"MRR@25 Score Final : {mrr_score:.5f}")
    print(f"Top-1  Accuracy   : {top1_hits / total * 100:.2f}% ({top1_hits}/{total})")
    print(f"Top-5  Accuracy   : {top5_hits / total * 100:.2f}% ({top5_hits}/{total})")
    print(f"Top-25 Accuracy   : {top25_hits / total * 100:.2f}% ({top25_hits}/{total})")
    print("="*50)
    
    return mrr_score

# Ejecutar evaluación sobre las moléculas de validación
mrr = compute_mrr_at_25_improved(
    query_mol_ids=unique_val_mols,
    mol_to_fp=mol_to_fp_val,
    mol_to_mz=val_precursor_mzs,
    ground_truth=val_ground_truth,
    candidate_fps=candidate_fps_matrix,
    candidate_mzs=candidate_mzs,
    candidate_df=candidate_df
)

Evaluando MRR@25 (Mejorado): 100%|██████████| 55162/55162 [01:40<00:00, 550.17it/s]


  RESULTADOS DE VALIDACIÓN LOCAL (MEJORADO - COCOS/PPM)  
MRR@25 Score Final : 0.15304
Top-1  Accuracy   : 6.97% (3844/55162)
Top-5  Accuracy   : 22.29% (12297/55162)
Top-25 Accuracy   : 59.45% (32792/55162)
